In [1]:
import sqlite3
import pandas as pd


In [2]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw" / "sample_superstore.csv").exists():
    ROOT = ROOT.parent
RAW_PATH = ROOT / "data" / "raw" / "sample_superstore.csv"
DB_PATH = ROOT / "database" / "sales.db"
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Sample Superstore dataset not found: {RAW_PATH}")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
connection = sqlite3.connect(DB_PATH)


In [3]:
df = pd.read_csv(RAW_PATH, encoding="latin1")

# Normalize column names and date/numeric fields before writing to SQLite.
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)
required_columns = {"Order_ID", "Customer_ID", "Customer_Name", "Order_Date", "Ship_Date", "Region", "Category", "Sub_Category", "Sales", "Quantity", "Discount", "Profit"}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Source dataset is missing required columns: {missing_columns}")
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Ship_Date"] = pd.to_datetime(df["Ship_Date"], errors="coerce")
for col in ["Sales", "Quantity", "Discount", "Profit"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["Order_ID", "Customer_ID", "Customer_Name", "Order_Date", "Sales", "Profit"]).drop_duplicates()
if df.empty:
    raise ValueError("No valid rows remain after source-data normalization.")
# Store ISO-8601 text so SQLite strftime() works consistently on every platform.
for col in ["Order_Date", "Ship_Date"]:
    df[col] = df[col].dt.strftime("%Y-%m-%d")


In [4]:
df.to_sql(
    "sales_data",
    connection,
    if_exists="replace",
    index=False
)


9994

In [5]:
test_query = "SELECT * FROM sales_data LIMIT 5"

pd.read_sql(test_query, connection)

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [6]:
query = """
SELECT sum(Sales) as total_revenue
FROM sales_Data
"""
pd.read_sql(query, connection)

,total_revenue
0,2.297201e+06


In [7]:
query= """
SELECT Category, SUM(sales) as revenue
FROM sales_data
GROUP BY Category
ORDER BY Revenue DESC
"""
pd.read_sql(query, connection)

,Category,revenue
0,Technology,836154.0330
1,Furniture,741999.7953
2,Office Supplies,719047.0320


In [8]:
query = """
SELECT
    Region,
    SUM(Sales) AS total_sales,
    AVG(Sales) AS avg_order_value
FROM sales_data
GROUP BY Region
ORDER BY total_sales DESC
"""
pd.read_sql(query, connection)

,Region,total_sales,avg_order_value
0,West,725457.8245,226.493233
1,East,678781.2400,238.336110
2,Central,501239.8908,215.772661
3,South,391721.9050,241.803645


In [9]:
query = """
SELECT
    Customer_Name,
    SUM(Sales) AS customer_revenue
FROM sales_data
GROUP BY Customer_Name
ORDER BY customer_revenue DESC
LIMIT 10
"""
pd.read_sql(query, connection)

,Customer_Name,customer_revenue
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
5,Ken Lonsdale,14175.229
6,Sanjit Chand,14142.334
7,Hunter Lopez,12873.298
8,Sanjit Engle,12209.438
9,Christopher Conant,12129.072


In [10]:
query = """
SELECT
    strftime('%Y-%m', Order_Date) AS month,
    SUM(Sales) AS monthly_revenue
FROM sales_data
GROUP BY month
ORDER BY month
"""
monthly_sql = pd.read_sql(query, connection)

monthly_sql.head()


,month,monthly_revenue
0,2014-01,14236.895
1,2014-02,4519.892
2,2014-03,55691.009
3,2014-04,28295.345
4,2014-05,23648.287
